# Avant le TP7 : un plan de réassignation écrit à la main, et les sorties JSON des outils

Le client Python n'a pas d'API pour réassigner des partitions ni pour lire les répertoires de données. Le TP7 pilote donc les outils Java depuis le notebook et exploite leurs sorties. Cette démonstration montre, sur un petit topic jetable :

1. comment lire la sortie JSON de `kafka-log-dirs.sh` en Python ;
2. le cycle complet d'une réassignation : plan, `--execute`, `--verify` ;
3. le lien entre l'ordre des répliques et le leader préféré.

Chaque étape montre la version **terminal**, puis la version **Python** quand elle existe. Une étape qui ne fait que lire exécute les deux. Une étape qui modifie le cluster n'en exécute qu'une : l'autre est affichée à côté, **à ne pas lancer en plus**.

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab.

Les noms et les valeurs sont définis une seule fois, ici. Dans un terminal de JupyterLab, définissez d'abord : `export TOPIC=demo-tp7 PLAN=/tmp/demo-plan.json`.

Création du topic, équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic $TOPIC --partitions 2 --replication-factor 3
```

**En Python**

In [1]:
import json
import os

from confluent_kafka import ElectionType, KafkaError, KafkaException, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}

TOPIC = "demo-tp7"
PARTITIONS = 2
RF = 3
PARTITION = 0  # partition dont on inverse les répliques
BROKER = 1  # broker dont on lit les répertoires de données
PLAN = "/tmp/demo-plan.json"  # dans le conteneur tools

admin = AdminClient(CONF)
try:
    admin.create_topics([NewTopic(TOPIC, num_partitions=PARTITIONS, replication_factor=RF)])[TOPIC].result()
    print(f"{TOPIC} créé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.TOPIC_ALREADY_EXISTS:
        raise
    print(f"{TOPIC} existe déjà")

demo-tp7 créé


Le placement de départ. **En terminal**

In [2]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic $TOPIC | grep Partition:

	Topic: demo-tp7	Partition: 0	Leader: 3	Replicas: 3,1,2	Isr: 3,1,2	Elr: 	LastKnownElr: 
	Topic: demo-tp7	Partition: 1	Leader: 1	Replicas: 1,2,3	Isr: 1,2,3	Elr: 	LastKnownElr: 


## 1. Une sortie JSON d'outil

`kafka-log-dirs.sh` affiche deux lignes de texte, puis tout son résultat sur **une seule ligne JSON**. Le client Python n'a pas d'équivalent : on lance l'outil, puis on exploite sa sortie en Python.

**En terminal**

In [3]:
!docker exec kafka-tools kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list $BROKER --topic-list $TOPIC

Querying brokers for log directories information


Received log directory information from brokers 1
{"brokers":[{"broker":1,"logDirs":[{"partitions":[{"partition":"demo-tp7-1","size":0,"offsetLag":0,"isFuture":false},{"partition":"demo-tp7-0","size":0,"offsetLag":0,"isFuture":false}],"error":null,"logDir":"/var/lib/kafka/data"}]}],"version":1}


**En Python** : `x = !command` range les lignes de la sortie dans une liste ; on garde la dernière, puis `json.loads` la transforme en dictionnaires. Si l'outil a échoué, sa dernière ligne n'est pas du JSON : on l'affiche au lieu de planter sur une erreur illisible.

In [4]:
output = !docker exec kafka-tools kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list $BROKER --topic-list $TOPIC
try:
    result = json.loads(output[-1])
except (IndexError, json.JSONDecodeError):
    raise RuntimeError("kafka-log-dirs.sh n'a pas rendu de JSON : %s" % output[-3:])

for broker in result["brokers"]:
    for log_dir in broker["logDirs"]:
        print("broker", broker["broker"], log_dir["logDir"],
              [(p["partition"], p["size"]) for p in log_dir["partitions"]])

broker 1 /var/lib/kafka/data [('demo-tp7-1', 0), ('demo-tp7-0', 0)]


## 2. Un plan écrit à la main

Un plan de réassignation est un fichier JSON : pour chaque partition, la liste des répliques voulues. Ici, on garde les mêmes brokers pour la partition 0, mais dans l'ordre inverse.

**En Python** : on lit les répliques actuelles et on construit le plan.

In [5]:
admin = AdminClient(CONF)
replicas = admin.list_topics(TOPIC, timeout=10).topics[TOPIC].partitions[PARTITION].replicas
plan = {"version": 1, "partitions": [{"topic": TOPIC, "partition": PARTITION, "replicas": list(reversed(replicas))}]}
print("avant :", replicas, "- plan :", json.dumps(plan))

avant : [3, 1, 2] - plan : {"version": 1, "partitions": [{"topic": "demo-tp7", "partition": 0, "replicas": [2, 1, 3]}]}


**En terminal** : le plan est écrit dans `/tmp` du conteneur `tools`. Le texte JSON est envoyé sur l'entrée de `tee`, qui l'écrit dans le fichier et le réaffiche. Dans une ligne `!`, `{json.dumps(plan)}` est remplacé par le texte du plan ; dans un terminal, on taperait ce texte.

In [6]:
!echo '{json.dumps(plan)}' | docker exec -i kafka-tools tee $PLAN

{"version": 1, "partitions": [{"topic": "demo-tp7", "partition": 0, "replicas": [2, 1, 3]}]}


Exécution du plan. Le client Python n'a pas d'API de réassignation : l'outil Java est la seule version.

**En terminal**

In [7]:
!docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file $PLAN --execute

Current partition replica assignment

{"version":1,"partitions":[{"topic":"demo-tp7","partition":0,"replicas":[3,1,2],"log_dirs":["/var/lib/kafka/data","/var/lib/kafka/data","/var/lib/kafka/data"]}]}

Save this to use as the --reassignment-json-file option during rollback


Successfully started partition reassignment for demo-tp7-0


`--execute` affiche d'abord la répartition actuelle : c'est le plan de retour arrière, à conserver. `--verify` relit le même fichier et dit où en est chaque partition. C'est aussi lui qui retire les limitations de débit posées par `--throttle`, qu'il y en ait ou non.

**En terminal**

In [8]:
!docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file $PLAN --verify

Status of partition reassignment:
Reassignment of partition demo-tp7-0 is completed.

Clearing broker-level throttles on brokers 1,2,3


Clearing topic-level throttles on topic demo-tp7


## 3. Ordre des répliques et leader préféré

Le **leader préféré** est le premier broker de la liste. La réassignation a changé l'ordre, mais pas encore le leader : il faut une élection.

**En terminal** : l'état avant l'élection.

In [9]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic $TOPIC | grep "Partition: $PARTITION"

	Topic: demo-tp7	Partition: 0	Leader: 3	Replicas: 2,1,3	Isr: 3,1,2	Elr: 	LastKnownElr: 


Élection, équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-leader-election.sh --bootstrap-server $BS --election-type PREFERRED \
  --topic $TOPIC --partition 0
```

**En Python** : `elect_leaders()` sur la seule partition concernée. Le résultat donne, par partition, `None` si l'élection a eu lieu, ou l'erreur. `ELECTION_NOT_NEEDED` veut dire que le leader préféré l'était déjà : ce n'est pas un échec.

In [10]:
def state():
    p = AdminClient(CONF).list_topics(TOPIC, timeout=10).topics[TOPIC].partitions[PARTITION]
    return "répliques %s, leader %s, préféré %s" % (p.replicas, p.leader, p.replicas[0])


print("après --verify :", state())

admin = AdminClient(CONF)
for tp, error in admin.elect_leaders(ElectionType.PREFERRED, [TopicPartition(TOPIC, PARTITION)]).result().items():
    if error is not None and error.code() != KafkaError.ELECTION_NOT_NEEDED:
        raise KafkaException(error)
    print("partition", tp.partition, ":", error.name() if error else "élection faite")
print("après élection :", state())

après --verify : répliques [2, 1, 3], leader 3, préféré 2


partition 0 : élection faite


après élection : répliques [2, 1, 3], leader 2, préféré 2


## Ménage

Équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --delete --topic $TOPIC
```

**En Python** pour le topic ; le fichier du plan, lui, se retire en terminal.

In [11]:
admin = AdminClient(CONF)
try:
    admin.delete_topics([TOPIC])[TOPIC].result()
    print(f"{TOPIC} supprimé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.UNKNOWN_TOPIC_OR_PART:
        raise
    print(f"{TOPIC} absent")

demo-tp7 supprimé


In [12]:
!docker exec kafka-tools rm -f $PLAN

## La méthode du TP7

Ouvrez `tps/jour2/tp7_exploitation/enonce.ipynb`.

- `execute()` et `verify_plan()` sont fournies ; vous écrivez `generate()`, qui exploite la sortie de `--generate`, puis `wait_until_done()`. Elles servent jusqu'à la fin du TP.
- Les cellules qui redémarrent des nœuds durent plusieurs minutes : exécutez-les une seule fois, sans les interrompre.
- Gardez Grafana ouvert (http://localhost:3000) pendant le rolling restart de l'étape 6.6 : les tuiles du contrôleur actif et des partitions sous-répliquées bougent à chaque nœud.